# Evaluation of repeated simulation experiments

Analysis of MLflow results for synthetic experiments repeated over independent
data-generation seeds. Each seed is retained as a separate observation; model
selection is performed independently within each seed and configuration.

Medical-data evaluation is intentionally handled in a separate notebook.

In [ ]:
import json
import os

import dotenv
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from scipy.stats import binomtest, ttest_1samp, wilcoxon
from statsmodels.stats.multitest import multipletests

from my_project.reporting import save_figure, save_latex_table, save_table
from my_project.reporting.presentation import (
    CATEGORICAL_PALETTE,
    SEQUENTIAL_PALETTE,
    configure_notebook_plots,
    graph_display_name,
    plot_label,
    uses_edge_count,
)

In [ ]:
dotenv.load_dotenv(override=True)
configure_notebook_plots()
pd.set_option("display.max_columns", None)

RESULTS_DIR = os.getenv("RESULTS_DIR")
RESULTS_LATEX_DIR = os.getenv("RESULTS_LATEX_DIR")
if not RESULTS_DIR:
    raise ValueError("RESULTS_DIR must be configured")

PALETTE = CATEGORICAL_PALETTE
PALETTE_SEQ = SEQUENTIAL_PALETTE
SIMULATION_EXPERIMENTS = ("higgs", "pairwise")
MAIN_DATA_SEED = "42"
GRAPH_VAR = "graph_name_main"
GRAPHLESS_MODELS = ["MLP", "XGBoost"]
EDGE_GRAPHS = ["ii", "ii_permuted", "empty", "fully_connected", "oracle"]

SPLIT = "test"  # "test" or "val"
METRIC = f"{SPLIT}/auc"
SELECT_METRIC = "val/auc"
TEST_METRICS = ["test/auc", "test/pr_auc"]
VAL_METRICS = ["val/auc", "val/pr_auc"]
REPORT_METRICS = VAL_METRICS + TEST_METRICS
METRIC_GROUPS = [
    ("val/auc", "test/auc"),
    ("val/pr_auc", "test/pr_auc"),
]

BEST_GROUP_COLUMNS = [
    "experiment_name",
    "model_cls",
    "graph_name_main",
    "graph_name",
    "threshold",
    "n_samples",
    "data_seed",
    "permutation_seed",
    "n_edges",
    "corr",
    "train_size",
]
BEST_GROUP_COLUMNS_WITHOUT_SEEDS = [
    column
    for column in BEST_GROUP_COLUMNS
    if column not in ["data_seed", "permutation_seed"]
]
TRIAL_GROUP_COLUMNS = BEST_GROUP_COLUMNS + ["trial_id"]
WINNER_KEY_COLUMNS = TRIAL_GROUP_COLUMNS
PAIR_KEYS = [
    "experiment_name",
    "model_cls",
    "threshold",
    "n_samples",
    "data_seed",
    "n_edges",
    "corr",
    "train_size",
]

NON_TUNED_PARAMS = {
    "n_samples",
    "positive_rate",
    "n_trials",
    "n_edges",
    "graph_n_edges",
    "best_n_estimators",
    "threshold",
    "trial_id",
    "data_seed",
    "train_size",
    "seed",
    "random_state",
}

HEATMAP_ROW = ["model_cls", GRAPH_VAR, "n_edges"]
HEATMAP_COL = ["train_size", "corr"]
MISSING_KEY_LABEL = "brak"
MAKE_LABEL_LABEL = "klasa modelu / nazwa grafu / liczba krawędzi"
RANK_COL = ("średnia ranga", "", "")
INDEX_COLS = ["model", "graph", "n_edges"]
_KEY_SENTINEL = "__MISSING__"

# Helper functions

Definitions are grouped before data loading so the remaining sections contain only preparation, analysis, and reporting steps.

In [ ]:
def config_trial_key(df, key_columns):
    """Build a stable string key while preserving missing grouping values."""
    keyed = df[key_columns].astype(object)
    keyed = keyed.where(df[key_columns].notna(), _KEY_SENTINEL)
    return keyed.astype(str).agg("|".join, axis=1)


def build_observed_grids(
    runs_df,
    parameter_columns,
    min_unique=2,
    exclude=NON_TUNED_PARAMS,
):
    """Infer tuning grids from parameter values recorded in MLflow."""
    grids = {}
    for parameter in parameter_columns:
        if parameter in exclude or parameter not in runs_df.columns:
            continue
        values = pd.to_numeric(runs_df[parameter], errors="coerce").dropna()
        unique_values = sorted(values.unique().tolist())
        if len(unique_values) >= min_unique:
            grids[parameter] = unique_values
    return grids


def classify_grid_position(value, grid):
    """Locate a value within the observed tuning grid."""
    grid = sorted(grid)
    if len(grid) <= 1:
        return "single"
    index = min(range(len(grid)), key=lambda position: abs(grid[position] - value))
    if index == 0:
        return "min-edge"
    if index == len(grid) - 1:
        return "max-edge"
    return "interior"


def grid_edge_summary(df, grids, edge_flag_fraction=0.5):
    """Summarize where winning configurations lie within tuning grids."""
    rows = []
    for parameter, grid in grids.items():
        if parameter not in df.columns:
            continue
        grid_sorted = sorted(grid)
        values = pd.to_numeric(df[parameter], errors="coerce").dropna()
        if values.empty:
            continue
        positions = values.apply(
            lambda value: classify_grid_position(value, grid_sorted)
        )
        fractions = positions.value_counts(normalize=True)
        fraction_min = float(fractions.get("min-edge", 0.0))
        fraction_max = float(fractions.get("max-edge", 0.0))
        suggestion = ""
        if fraction_min >= edge_flag_fraction:
            suggestion = f"↓ rozszerz w dół (< {grid_sorted[0]:g})"
        elif fraction_max >= edge_flag_fraction:
            suggestion = f"↑ rozszerz w górę (> {grid_sorted[-1]:g})"
        rows.append(
            {
                "param": parameter,
                "explored_values": grid_sorted,
                "n_explored": len(grid_sorted),
                "n_configs": int(len(values)),
                "modal_value": values.mode().iloc[0],
                "pct_min_edge": fraction_min,
                "pct_interior": float(fractions.get("interior", 0.0)),
                "pct_max_edge": fraction_max,
                "suggest_expand": suggestion,
            }
        )
    return pd.DataFrame(rows)


def grid_edge_report(row, grids):
    """Return an edge diagnostic for one winning configuration."""
    records = []
    for parameter, grid in grids.items():
        if parameter not in row.index or pd.isna(row[parameter]):
            continue
        value = float(row[parameter])
        grid_sorted = sorted(grid)
        position = classify_grid_position(value, grid_sorted)
        records.append(
            {
                "param": parameter,
                "best_value": value,
                "explored_min": grid_sorted[0],
                "explored_max": grid_sorted[-1],
                "position": position,
                "on_edge": position in ("min-edge", "max-edge"),
            }
        )
    return pd.DataFrame(records)

In [ ]:
def panel_label(column):
    """Return a translated facet label for a metric column."""
    if "/" in column:
        return plot_label(column.split("/", 1)[0])
    return plot_label(column)


def metric_name(column):
    """Return the translated metric name after the final slash."""
    return plot_label(column.rsplit("/", 1)[-1])


def draw_heatmap(data, *, heatmap_row, heatmap_col, **kwargs):
    """Pivot long-form metrics and draw one heatmap panel."""
    pivot = data.pivot_table(
        index=heatmap_row,
        columns=heatmap_col,
        values="value",
        aggfunc="max",
    )

    def numeric_key(column):
        values = column if isinstance(column, tuple) else (column,)
        return tuple(float(value) for value in values)

    try:
        pivot = pivot[sorted(pivot.columns, key=numeric_key)]
    except (ValueError, TypeError):
        pass
    sns.heatmap(pivot, **kwargs)


def plot_metric_heatmaps(
    df,
    *,
    heatmap_row,
    heatmap_col,
    metric_groups,
    title=None,
):
    """Draw one heatmap figure for each group of metric columns."""
    heatmap_row = list(heatmap_row)
    heatmap_col = list(heatmap_col)
    id_columns = list(dict.fromkeys(heatmap_row + heatmap_col))

    heatmap_df = df.copy()
    graphless_mask = heatmap_df["model_cls"].isin(GRAPHLESS_MODELS)
    heatmap_df.loc[graphless_mask, GRAPH_VAR] = "—"
    heatmap_df.loc[graphless_mask, "conv_layer"] = pd.NA
    heatmap_df["model"] = heatmap_df["model_cls"].where(
        heatmap_df["conv_layer"].isna(),
        heatmap_df["model_cls"],
    )

    for column in id_columns:
        heatmap_df[column] = (
            heatmap_df[column]
            .astype(object)
            .where(heatmap_df[column].notna(), MISSING_KEY_LABEL)
        )

    for metric_columns in metric_groups:
        present = [column for column in metric_columns if column in heatmap_df.columns]
        missing = [
            column for column in metric_columns if column not in heatmap_df.columns
        ]
        if missing:
            print(f"Skipping columns missing from the data: {missing}")
        if not present:
            continue

        long_df = heatmap_df.melt(
            id_vars=id_columns,
            value_vars=present,
            var_name="metric",
            value_name="value",
        )
        long_df["panel"] = long_df["metric"].map(panel_label)
        value_min = np.nanmin(long_df["value"])
        value_max = np.nanmax(long_df["value"])
        panel_order = list(dict.fromkeys(panel_label(column) for column in present))

        grid = sns.FacetGrid(
            long_df,
            col="panel",
            col_order=panel_order,
            height=4,
            aspect=1.3,
        )
        grid.map_dataframe(
            draw_heatmap,
            heatmap_row=heatmap_row,
            heatmap_col=heatmap_col,
            annot=True,
            fmt=".3f",
            cmap="viridis",
            vmin=value_min,
            vmax=value_max,
            cbar=False,
        )
        grid.set_titles(col_template="{col_name}")
        grid.set_axis_labels(
            " / ".join(plot_label(column) for column in heatmap_col),
            " / ".join(plot_label(row) for row in heatmap_row),
        )
        for axis in grid.axes.flat:
            axis.tick_params(axis="x", rotation=90)

        group_title = " / ".join(
            dict.fromkeys(metric_name(column) for column in present)
        )
        figure_title = (
            group_title if title is None else f"{plot_label(title)} — {group_title}"
        )
        grid.figure.suptitle(figure_title, y=1.02)
        grid.figure.subplots_adjust(right=0.9)
        colorbar_axis = grid.figure.add_axes([0.92, 0.15, 0.02, 0.7])
        scalar_mappable = ScalarMappable(
            norm=Normalize(vmin=value_min, vmax=value_max),
            cmap="viridis",
        )
        grid.figure.colorbar(scalar_mappable, cax=colorbar_axis)
        plt.show()

In [ ]:
def numeric_sorted(values):
    """Sort unique non-missing values numerically when possible."""
    values = list(pd.Series(values).dropna().unique())
    try:
        return sorted(values, key=float)
    except (ValueError, TypeError):
        return sorted(values)


def make_label(row):
    """Build the model, graph, and edge-count label used in plots."""
    parts = [str(row["model_cls"])]
    if row["model_cls"] not in GRAPHLESS_MODELS:
        graph_name = row[GRAPH_VAR]
        parts.append(graph_display_name(graph_name))
        if uses_edge_count(graph_name):
            parts.append(str(row["n_edges"]))
    return " / ".join(parts)


def label_order(df, metric=METRIC):
    """Order model labels by mean rank across experimental settings."""
    keyed = df.copy()
    keyed["train_size"] = keyed["train_size"].astype(int)
    # keyed = keyed[keyed["train_size"] == keyed["train_size"].max()]
    graphless = keyed["model_cls"].isin(GRAPHLESS_MODELS)
    has_edge_count = (~graphless) & keyed[GRAPH_VAR].map(uses_edge_count)
    keyed["model"] = keyed["model_cls"].astype(str)
    keyed["graph"] = keyed[GRAPH_VAR].astype(str).where(~graphless, "")
    keyed["n_edges"] = keyed["n_edges"].astype(str).where(has_edge_count, "")
    keyed["_label"] = keyed.apply(make_label, axis=1)

    index_columns = ["model", "graph", "n_edges"]
    means = (
        keyed.groupby(index_columns + ["corr", "train_size"], dropna=False)[metric]
        .mean()
        .unstack(["corr", "train_size"])
    )
    ranks = means.rank(axis=0, ascending=False, method="min").mean(axis=1)
    ranked = (
        ranks.rename("_mean_rank")
        .reset_index()
        .sort_values(["_mean_rank"] + index_columns)
    )
    label_map = keyed.drop_duplicates(index_columns).set_index(index_columns)["_label"]
    return [label_map.loc[tuple(row[index_columns])] for _, row in ranked.iterrows()]


def plot_faceted_boxes(
    df,
    *,
    experiment_name,
    metric,
    x,
    order,
    x_label,
    filename=None,
    font_size=14,
    catplot_kwargs=None,
):

    if catplot_kwargs is None:
        catplot_kwargs = {
            "height": 5,
            "aspect": 1.6,
        }

    """Draw and save faceted boxplots with black crosses marking means."""
    df = df.copy()
    row_variable, column_variable = (
        ("corr", "train_size")
        if experiment_name == "pairwise"
        else ("train_size", "corr")
    )
    for facet_variable in (row_variable, column_variable):
        df[facet_variable] = df[facet_variable].fillna(-1).astype(str)

    row_order = numeric_sorted(df[row_variable])
    column_order = numeric_sorted(df[column_variable])
    grid = sns.catplot(
        data=df,
        kind="box",
        x=x,
        y=metric,
        hue=x,
        order=order,
        hue_order=order,
        col=column_variable,
        row=row_variable,
        row_order=row_order,
        col_order=column_order,
        legend=False,
        sharex=True,
        sharey="row",
        palette=PALETTE,
        margin_titles=True,
        **catplot_kwargs,
    )
    grid.map_dataframe(
        sns.pointplot,
        x=x,
        y=metric,
        hue=x,
        order=order,
        hue_order=order,
        estimator=np.mean,
        errorbar=None,
        dodge=0,
        markers="x",
        linestyles="none",
        markersize=5,
        markeredgewidth=0.6,
        palette={label: "black" for label in order},
        legend=False,
        zorder=10,
    )

    show_rows = len(row_order) > 1
    show_columns = len(column_order) > 1
    grid.set_titles(
        row_template="{row_name}" if show_rows else "",
        col_template="{col_name}" if show_columns else "",
        size=font_size,
    )
    grid.set_axis_labels("", "")

    row_count, column_count = grid.axes.shape
    if show_columns:
        axis = grid.axes[0, (column_count - 1) // 2]
        axis.set_title(
            f"{plot_label(column_variable)}\n{axis.get_title()}",
            fontsize=font_size,
        )
    if show_rows and grid._margin_titles_texts:
        title = grid._margin_titles_texts[(row_count - 1) // 2]
        title.set_text(f"{plot_label(row_variable)}\n\n{title.get_text()}")
        title.set_multialignment("center")
    for title in grid._margin_titles_texts:
        title.set_fontsize(font_size)
    for axis in grid.axes.flat:
        axis.tick_params(axis="both", labelsize=font_size)
        axis.tick_params(axis="x", labelrotation=90)

    figure = grid.figure
    grid.tight_layout(rect=[0.03, 0.03, 0.98, 0.98])
    boxes = [axis.get_position() for axis in grid.axes.flat]
    left, right = min(box.x0 for box in boxes), max(box.x1 for box in boxes)
    bottom, top = min(box.y0 for box in boxes), max(box.y1 for box in boxes)
    figure.supxlabel(x_label, x=(left + right) / 2, y=0.005, fontsize=font_size)
    figure.supylabel(
        metric,
        x=0.005,
        y=(bottom + top) / 2,
        fontsize=font_size,
    )
    if filename:
        save_figure(figure, filename)
    plt.show()

In [ ]:
def label_parts(experiment_df):
    """Split combined labels into model, graph, and edge-count columns."""
    df = experiment_df.copy()
    graphless = df["model_cls"].isin(GRAPHLESS_MODELS)
    has_edge_count = (~graphless) & df["graph_name_main"].map(uses_edge_count)
    df["model"] = df["model_cls"].astype(str)
    df["graph"] = df["graph_name_main"].map(graph_display_name).where(~graphless, "")
    df["n_edges"] = df["n_edges"].astype(str).where(has_edge_count, "")
    return df


def make_summary_table(experiment_df, metric):
    """Create a mean, standard deviation, and rank table by configuration."""
    experiment_df = label_parts(experiment_df)
    summary_table = experiment_df.pivot_table(
        index=INDEX_COLS,
        columns=["corr", "train_size"],
        values=metric,
        aggfunc=["mean", "std"],
        dropna=False,
    )
    summary_table = summary_table.dropna(axis=0, how="all")
    summary_table = summary_table.swaplevel(0, -1, axis=1).swaplevel(0, 1, axis=1)
    summary_table = summary_table.sort_index(axis=1, level=["corr", "train_size"])
    summary_table.columns = summary_table.columns.set_names(
        ["corr", "train_size", "stat"]
    )

    mean_df = summary_table.xs("mean", axis=1, level="stat")
    rank_df = mean_df.rank(axis=0, ascending=False, method="min").astype("Int64")
    rank_df.columns = pd.MultiIndex.from_tuples(
        [(corr, train_size, "rank") for corr, train_size in rank_df.columns],
        names=summary_table.columns.names,
    )
    summary_table = pd.concat([summary_table, rank_df], axis=1)

    def configuration_key(configuration):
        corr, train_size = configuration
        try:
            return float(corr), float(train_size)
        except (ValueError, TypeError):
            return corr, train_size

    ordered_columns = [
        (corr, train_size, statistic)
        for corr, train_size in sorted(mean_df.columns, key=configuration_key)
        for statistic in ("mean", "std", "rank")
    ]
    summary_table = summary_table[ordered_columns]
    summary_table[RANK_COL] = rank_df.droplevel("stat", axis=1).mean(axis=1)
    return summary_table.sort_values([RANK_COL, "model", "graph", "n_edges"])

In [ ]:
def paired_ii_vs_permuted(df, metric=METRIC):
    """Return matched II and mean-permuted scores for each configuration."""
    graph_scores = {
        name: (
            df[df["graph_name_main"] == graph]
            .groupby(PAIR_KEYS, dropna=False)[metric]
            .mean()
            .rename(name)
        )
        for name, graph in {
            "score_ii": "ii",
            "score_perm": "ii_permuted",
        }.items()
    }
    paired = pd.concat(graph_scores.values(), axis=1).dropna().reset_index()
    paired["diff"] = paired["score_ii"] - paired["score_perm"]
    return paired


def test_differences(differences, test, test_kwargs=None):
    """Test the one-sided hypothesis that paired differences are positive."""
    differences = differences.dropna()
    if test == "t":
        kwargs = {"alternative": "greater"} if test_kwargs is None else test_kwargs
        if len(differences) < 2 or differences.std(ddof=1) == 0:
            return float("nan"), float("nan")
        result = ttest_1samp(differences, 0, **kwargs)
    elif test == "wilcoxon":
        kwargs = (
            {"alternative": "greater", "method": "exact"}
            if test_kwargs is None
            else test_kwargs
        )
        try:
            result = wilcoxon(differences, **kwargs)
        except ValueError:
            return float("nan"), float("nan")
    elif test == "sign":
        nonzero = differences[differences != 0]
        if nonzero.empty:
            return float("nan"), float("nan")
        kwargs = {"alternative": "greater"} if test_kwargs is None else test_kwargs
        positive_count = int((nonzero > 0).sum())
        result = binomtest(positive_count, len(nonzero), p=0.5, **kwargs)
        return positive_count, result.pvalue
    else:
        raise ValueError("test must be 't', 'wilcoxon', or 'sign'")
    return result.statistic, result.pvalue


def paired_test_summary(
    df,
    metric=METRIC,
    test="wilcoxon",
    group_columns=("experiment_name",),
    alpha=0.05,
    method="holm",
    test_kwargs=None,
):
    """Summarize paired II-versus-permuted tests with multiplicity correction."""
    group_columns = list(group_columns)
    paired = paired_ii_vs_permuted(df, metric)
    count_name = "n_pairs"

    groups = (
        paired.groupby(group_columns, dropna=False) if group_columns else [((), paired)]
    )
    rows = []
    for values, subset in groups:
        values = values if isinstance(values, tuple) else (values,)
        statistic, p_value = test_differences(
            subset["diff"], test, test_kwargs=test_kwargs
        )
        rows.append(
            {
                **dict(zip(group_columns, values)),
                count_name: len(subset),
                "median_ii": subset["score_ii"].median(),
                "median_perm": subset["score_perm"].median(),
                "median_diff": subset["diff"].median(),
                "stat": statistic,
                "p_value": p_value,
            }
        )

    result = pd.DataFrame(rows)
    if result.empty:
        return result

    result["p_rank"] = pd.Series(pd.NA, index=result.index, dtype="Int64")
    result["alpha_corrected"] = np.nan
    result["p_m"] = np.nan
    result["passed"] = False
    correction_groups = (
        result.groupby("experiment_name", dropna=False).groups.values()
        if "experiment_name" in result
        else [result.index]
    )
    for indexes in correction_groups:
        valid = result.loc[indexes, "p_value"].dropna().sort_values()
        if not valid.empty:
            ranks = np.arange(1, len(valid) + 1)
            rejected, corrected_p, _, _ = multipletests(
                valid,
                alpha=alpha,
                method=method,
            )
            result.loc[valid.index, "p_rank"] = ranks
            # corrected alpha for Holm method
            result.loc[valid.index, "alpha_corrected"] = alpha / (
                len(valid) - ranks + 1
            )
            result.loc[valid.index, "p_m"] = corrected_p
            result.loc[valid.index, "passed"] = rejected

    sort_columns = [
        column for column in ["experiment_name", "p_value"] if column in result
    ]
    return result.sort_values(sort_columns, na_position="last").reset_index(drop=True)

# Data preparation

Load the exported MLflow snapshot, normalize its schema, and select one winning trial independently for each simulation repeat.

In [ ]:
csv_path = os.path.join(RESULTS_DIR, "mlflow_selected_runs_df.csv")
schema_path = os.path.join(RESULTS_DIR, "mlflow_selected_runs_schema.json")
selected_runs_df = pd.read_csv(csv_path, dtype=str)

if os.path.exists(schema_path):
    with open(schema_path, encoding="utf-8") as schema_file:
        column_schema = json.load(schema_file)
else:
    column_schema = {}

PARAM_COLUMNS = column_schema.get(
    "parameter_columns",
    [
        column.split(".", 1)[1]
        for column in selected_runs_df.columns
        if column.startswith("params.")
    ],
)
METRIC_COLUMNS = column_schema.get(
    "metric_columns",
    [
        column.split(".", 1)[1]
        for column in selected_runs_df.columns
        if column.startswith("metrics.")
    ],
)

# Prefer existing short names; prefixed variants only fill missing values in older exports.
column_order = sorted(
    range(len(selected_runs_df.columns)),
    key=lambda index: "." in selected_runs_df.columns[index],
)
selected_runs_df = selected_runs_df.iloc[:, column_order]
selected_runs_df.columns = [
    column.split(".", 1)[1] if "." in column else column
    for column in selected_runs_df.columns
]
selected_runs_df = selected_runs_df.T.groupby(level=0, sort=False).first().T
metric_column_names = {
    "val/avg_precision": "val/pr_auc",
    "test/avg_precision": "test/pr_auc",
}
selected_runs_df = selected_runs_df.rename(columns=metric_column_names)
METRIC_COLUMNS = [metric_column_names.get(column, column) for column in METRIC_COLUMNS]
selected_runs_df = selected_runs_df[
    selected_runs_df["experiment_name"].isin(SIMULATION_EXPERIMENTS)
].copy()
if "cov" in selected_runs_df.columns:
    if "corr" in selected_runs_df.columns:
        corr = selected_runs_df["corr"].fillna(selected_runs_df["cov"])
        selected_runs_df = selected_runs_df.drop(columns=["corr", "cov"])
        selected_runs_df = selected_runs_df.join(corr.rename("corr"))
    else:
        selected_runs_df = selected_runs_df.rename(columns={"cov": "corr"})

metric_columns = [
    column for column in METRIC_COLUMNS if column in selected_runs_df.columns
]
selected_runs_df[metric_columns] = selected_runs_df[metric_columns].apply(
    pd.to_numeric,
    errors="coerce",
)
for column in ["graph_precision", "graph_recall"]:
    if column in selected_runs_df.columns:
        selected_runs_df[column] = pd.to_numeric(
            selected_runs_df[column],
            errors="coerce",
        )

seed_counts = selected_runs_df.groupby("experiment_name")["data_seed"].nunique()
missing_experiments = set(SIMULATION_EXPERIMENTS) - set(seed_counts.index)
insufficient_repeats = seed_counts[seed_counts < 2]
if missing_experiments or not insufficient_repeats.empty:
    raise ValueError(
        "Expected repeated simulation runs for every configured experiment; "
        f"missing={sorted(missing_experiments)}, "
        f"seed_counts={insufficient_repeats.to_dict()}"
    )
EXPERIMENTS = [
    experiment
    for experiment in SIMULATION_EXPERIMENTS
    if experiment in selected_runs_df["experiment_name"].unique()
]

graph_names = selected_runs_df["graph_name"].astype("string")
is_permuted = graph_names.str.contains("permuted", na=False)
graph_name_main = graph_names.where(
    ~is_permuted,
    graph_names.str.split("_permuted_").str[0] + "_permuted",
)
permutation_seed = graph_names.str.split("_permuted_").str[1].where(is_permuted)
selected_runs_df["model_cls"] = selected_runs_df["model_cls"].str.replace(
    "^My",
    "",
    regex=True,
)
selected_runs_df = selected_runs_df.drop(
    columns=["is_permuted", "graph_name_main", "permutation_seed"],
    errors="ignore",
)
selected_runs_df = pd.concat(
    [
        selected_runs_df,
        pd.DataFrame(
            {
                "is_permuted": is_permuted,
                "graph_name_main": graph_name_main,
                "permutation_seed": permutation_seed,
            },
            index=selected_runs_df.index,
        ),
    ],
    axis=1,
)
seed_counts

In [ ]:
metric_families = ["val", "test"]
reportable_metric_columns = [
    column
    for family in metric_families
    for column in sorted(selected_runs_df.columns)
    if column.startswith(f"{family}/")
]
reportable_metric_columns += ["graph_precision", "graph_recall"]

mean_columns = [
    column for column in METRIC_COLUMNS if column in selected_runs_df.columns
]
mean_columns += [
    column
    for column in ["graph_precision", "graph_recall"]
    if column in selected_runs_df.columns
]

# Collapse duplicate records for each Optuna trial before selecting by validation AUC.
aggregation = {column: "mean" for column in mean_columns}
for column in selected_runs_df.columns:
    if column not in aggregation and column not in TRIAL_GROUP_COLUMNS:
        aggregation[column] = "first"

per_trial = selected_runs_df.groupby(
    TRIAL_GROUP_COLUMNS,
    dropna=False,
    as_index=False,
).agg(aggregation)
best_per_config = (
    per_trial.dropna(subset=[SELECT_METRIC])
    .sort_values(SELECT_METRIC, ascending=False)
    .drop_duplicates(subset=BEST_GROUP_COLUMNS, keep="first")
    .sort_values(METRIC, ascending=False)
    .reset_index(drop=True)
)

label_columns = BEST_GROUP_COLUMNS + ["conv_layer", "mlflow.runName"]
best_per_config_columns = [
    column
    for column in label_columns + reportable_metric_columns
    if column in best_per_config.columns
]
best_per_config[best_per_config_columns]

In [ ]:
winner_keys = set(config_trial_key(best_per_config, WINNER_KEY_COLUMNS))
best_per_config_runs = (
    selected_runs_df[
        config_trial_key(selected_runs_df, WINNER_KEY_COLUMNS).isin(winner_keys)
    ]
    .copy()
    .reset_index(drop=True)
)

print(f"Selected configurations: {len(best_per_config)}")
print(f"Source simulation runs: {len(best_per_config_runs)}")
print(
    best_per_config_runs.groupby("experiment_name")["data_seed"]
    .nunique()
    .rename("n_data_seeds")
)

# Hyperparameter grid edge diagnostics

For each winning config (`best_per_config`) we check where, relative to the
**values actually tried during tuning** (distinct realizations from
`selected_runs_df`), the best value of a given hyperparameter landed:

- **min-edge / max-edge** - the value sits at the edge of the searched range,
  suggesting the optimum may be *outside* what Optuna managed to try and
  it is worth extending the range in that direction.
- **interior** - the optimum is inside the searched range, so the range is
  probably sufficient.

The grid is not hardcoded; it is derived from the actual parameter values
recorded in the MLflow runs.

In [ ]:
OBSERVED_GRIDS = build_observed_grids(
    selected_runs_df,
    PARAM_COLUMNS,
)
edge_summary = grid_edge_summary(
    best_per_config[best_per_config["model_cls"] == "MLP"],
    OBSERVED_GRIDS,
)
edge_summary

In [ ]:
best_row = best_per_config.iloc[0]
print(
    f"Best run: {best_row.get('model_cls', '?')} / "
    f"{best_row.get('graph_name', '?')} — {METRIC} = {best_row[METRIC]:.4f}"
)
grid_edge_report(best_row, OBSERVED_GRIDS)

# Heatmaps

In [ ]:
for experiment_name in EXPERIMENTS:
    experiment_df = best_per_config[
        best_per_config["experiment_name"] == experiment_name
    ]
    if experiment_df.empty:
        continue
    plot_metric_heatmaps(
        experiment_df,
        heatmap_row=HEATMAP_ROW,
        heatmap_col=HEATMAP_COL,
        metric_groups=METRIC_GROUPS,
        title=experiment_name,
    )

## Box plots of metrics by model class, graph, and sample size

In [ ]:
# Each observation is the winning run for one independently generated dataset.
plot_df = best_per_config_runs[
    best_per_config_runs["data_seed"].astype(str) != MAIN_DATA_SEED
].copy()
grid_df = plot_df.copy()
grid_df["_label"] = grid_df.apply(make_label, axis=1)
grid_df["_hue"] = grid_df["_label"]

plot_df.groupby("experiment_name")["data_seed"].nunique().rename("n_plot_seeds")

In [ ]:
catplot_kwargs_dict = {
    "higgs": {"height": 5, "aspect": 1.6},
    "pairwise": {"height": 8, "aspect": 1.4},
}

font_size_dict = {
    "higgs": 14,
    "pairwise": 30,
}

In [ ]:
# for experiment_name in EXPERIMENTS:
for experiment_name in ["pairwise"]:
    experiment_df = grid_df[grid_df["experiment_name"] == experiment_name]
    if experiment_df.empty:
        continue

    for requested_metric in REPORT_METRICS:
        metric = requested_metric
        if metric not in experiment_df or experiment_df[metric].notna().sum() == 0:
            fallback = metric.replace("test/", "val/", 1)
            if (
                fallback == metric
                or fallback not in experiment_df
                or not experiment_df[fallback].notna().any()
            ):
                print(f"Skipping {experiment_name} / {metric}: no values available.")
                continue
            metric = fallback
            print(f"{experiment_name}: {requested_metric} unavailable; using {metric}.")

        order = label_order(experiment_df, metric)
        plot_faceted_boxes(
            experiment_df,
            experiment_name=experiment_name,
            metric=metric,
            x="_label",
            order=order,
            x_label=MAKE_LABEL_LABEL,
            filename=f"evaluation__boxplot__{experiment_name}__{metric}",
            catplot_kwargs=catplot_kwargs_dict[experiment_name],
            font_size=font_size_dict[experiment_name],
        )

In [ ]:
# Compare the original II graph with its permutations across data-generation seeds.
facet_cols = ["corr", "train_size"]
permuted_df = grid_df[grid_df["is_permuted"]].copy()

for experiment_name in EXPERIMENTS:
    permuted_experiment_df = permuted_df[
        permuted_df["experiment_name"] == experiment_name
    ].copy()

    selection = (
        permuted_experiment_df.groupby(facet_cols + ["n_edges"], dropna=False)
        .agg(
            selection_score=("val/auc", "mean"),
            n_data_seeds=("data_seed", "nunique"),
        )
        .reset_index()
        .query("n_data_seeds > 1")
        .sort_values("selection_score", ascending=False)
        .drop_duplicates(facet_cols)
    )
    experiment_df = grid_df[
        (grid_df["experiment_name"] == experiment_name)
        & grid_df["graph_name_main"].isin(["ii", "ii_permuted"])
    ].merge(
        selection[facet_cols + ["n_edges"]],
        on=facet_cols + ["n_edges"],
        how="inner",
    )
    if experiment_df.empty:
        continue

    experiment_df["permutation_seed"] = pd.to_numeric(
        experiment_df["permutation_seed"], errors="coerce"
    ).astype("Int64")
    permuted_mask = experiment_df["graph_name_main"] == "ii_permuted"
    experiment_df["_permutation_label"] = "ii"
    experiment_df.loc[permuted_mask, "_permutation_label"] = (
        "ziarno permutacji "
        + experiment_df.loc[permuted_mask, "permutation_seed"].astype(str)
    )
    order = ["ii"] + [
        f"ziarno permutacji {seed}"
        for seed in numeric_sorted(experiment_df.loc[permuted_mask, "permutation_seed"])
    ]

    for metric in REPORT_METRICS:
        if metric not in experiment_df or not experiment_df[metric].notna().any():
            continue
        plot_faceted_boxes(
            experiment_df,
            experiment_name=experiment_name,
            metric=metric,
            x="_permutation_label",
            order=order,
            x_label="graf / ziarno permutacji",
            filename=f"evaluation__permuted_graphs__{experiment_name}__{metric}",
        )

# Summary tables

Aggregate each model and graph configuration across repeated simulations, rank it within every experimental setting, and export the report tables.

In [ ]:
for experiment_name in EXPERIMENTS:
    for metric in REPORT_METRICS:
        experiment_df = grid_df[grid_df["experiment_name"] == experiment_name]
        if experiment_df.empty:
            print(f"Skipping {experiment_name} because no runs were found.")
            continue

        summary_table = make_summary_table(experiment_df, metric)
        title = f"{experiment_name}__{metric}"
        summary_table_export, summary_table_path = save_latex_table(
            summary_table,
            "evaluation__" + title,
            value_decimals=3,
            std_scale=1000,
            std_decimals=2,
        )

summary_table_export

In [ ]:
experiment_name = "pairwise"
metric = "test/auc"

experiment_df = grid_df[grid_df["experiment_name"] == experiment_name]
summary_table = make_summary_table(experiment_df, metric)

overall_rank = summary_table[RANK_COL]

summary_table = (
    summary_table.drop(columns=[RANK_COL])
    .stack(level="corr", future_stack=True)
    .reorder_levels(["corr", *INDEX_COLS])
    .sort_index()
)


rank_columns = summary_table.xs("rank", axis=1, level="stat")
summary_table[("średnia ranga cov", "")] = rank_columns.mean(axis=1)

summary_table[("średnia ranga", "")] = overall_rank.reindex(
    summary_table.index.droplevel("corr")
).to_numpy()

summary_table = summary_table.sort_values(by=["corr", "średnia ranga"], ascending=True)

summary_table_export, summary_table_path = save_latex_table(
    summary_table,
    f"evaluation__{experiment_name}__{metric}__horizontal",
    value_decimals=3,
    std_scale=1000,
    std_decimals=2,
)

summary_table_export

# Paired test: ii vs mean ii_permuted

For each simulation configuration and `data_seed`, pair the score of the `ii`
graph with the mean score of its `ii_permuted_*` counterparts. Statistical tests
then compare these paired differences across independent data-generation seeds.
The pooled variant first averages correlated configurations within each seed, so
each seed contributes exactly one observation.

In [ ]:
df_test = best_per_config[best_per_config["data_seed"] != MAIN_DATA_SEED]

In [ ]:
TEST = "sign"
# TEST = "t"
# TEST = "wilcoxon"

# p > 0.5
test_kwargs = {"alternative": "greater"}

In [ ]:
for experiment in EXPERIMENTS:
    df_test_i = df_test[df_test["experiment_name"] == experiment]
    summary_paired_ = (
        paired_test_summary(
            df_test_i,
            METRIC,
            TEST,
            test_kwargs=test_kwargs,
            group_columns=("experiment_name", "corr", "train_size", "n_edges"),
            alpha=0.05,
        )
        .sort_values(["p_value", "p_rank"])
        .reset_index(drop=True)
    )
    print(experiment)
    display(summary_paired_)

    summary_paired_["stat"] = summary_paired_["stat"].astype(int)

    summary_paired_export = summary_paired_[
        [
            "corr",
            "train_size",
            "n_edges",
            # "median_ii",
            # "median_perm",
            # "median_diff",
            "stat",
            "p_value",
            "p_rank",
            "alpha_corrected",
            # "p_m",
            "passed",
        ]
    ]
    if experiment == "higgs":
        summary_paired_export = summary_paired_export.drop(columns=["corr"])

    summary_paired_export["p_value"] = summary_paired_export["p_value"].map(
        lambda value: f"{value:.3e}" if pd.notna(value) else ""
    )
    summary_paired_export = summary_paired_export.round(4)

    summary_paired_export = summary_paired_export.round(4)
    summary_paired_export["passed"] = summary_paired_export["passed"].map(
        {True: "TAK", False: "NIE"}
    )
    save_table(
        summary_paired_export,
        f"evaluation__paired_test_summary__{experiment}__{METRIC}",
        index=False,
        float_format="%.4f",
    )
    display(summary_paired_export)

## Boxplots of paired differences

Each box shows the distribution of paired differences $\mathrm{AUC}_{ii} - \mathrm{AUC}_{ii\_permuted}$ within the same `experiment_name`, `corr`, `train_size`, and `n_edges` groups used by the statistical tests.

In [ ]:
paired_plot_df = paired_ii_vs_permuted(df_test, METRIC)
difference_label = f"różnica {METRIC}: ii - ii_permuted"
paired_plot_df[difference_label] = paired_plot_df["diff"]

for experiment in EXPERIMENTS:
    experiment_df = paired_plot_df[
        paired_plot_df["experiment_name"] == experiment
    ].copy()
    if experiment_df.empty:
        continue

    edge_order = numeric_sorted(experiment_df["n_edges"])
    plot_faceted_boxes(
        experiment_df,
        experiment_name=experiment,
        metric=difference_label,
        x="n_edges",
        order=edge_order,
        x_label=plot_label("n_edges"),
        filename=f"evaluation__paired_differences__{experiment}__{METRIC}",
    )

## Descriptive statistics of paired differences

Mean, median, sample skewness, and quartile-based Bowley skewness of $\mathrm{AUC}_{ii} - \mathrm{AUC}_{ii\_permuted}$ are calculated within the same groups used by the statistical tests. Bowley skewness is reported as missing when the interquartile range is zero.

In [ ]:
def bowley_skewness(values):
    """Return quartile-based skewness, or NaN when the IQR is zero."""
    q1, median, q3 = values.quantile([0.25, 0.5, 0.75])
    iqr = q3 - q1
    if iqr == 0:
        return float("nan")
    return (q3 + q1 - 2 * median) / iqr


difference_statistics = (
    paired_plot_df.groupby(
        ["experiment_name", "corr", "train_size", "n_edges"],
        dropna=False,
    )["diff"]
    .agg(
        n_pairs="count",
        mean="mean",
        median="median",
        skewness="skew",
        bowley_skewness=bowley_skewness,
    )
    .reset_index()
    .sort_values(
        ["experiment_name", "corr", "train_size", "n_edges"], na_position="first"
    )
    .reset_index(drop=True)
)

display(difference_statistics.round(4))

for experiment in EXPERIMENTS:
    statistics_export = difference_statistics[
        difference_statistics["experiment_name"] == experiment
    ].drop(columns=["experiment_name"])

    statistics_export = statistics_export[
        [
            "corr",
            "train_size",
            "n_edges",
            "n_pairs",
            "mean",
            "median",
            "skewness",
            "bowley_skewness",
        ]
    ]

    if experiment == "higgs":
        statistics_export = statistics_export.drop(columns=["corr"])

    save_table(
        statistics_export.round(4),
        f"evaluation__paired_difference_statistics__{experiment}__{METRIC}",
        index=False,
        float_format="%.4f",
    )